# Análise Exploratória de Dados — Datathon Passos Mágicos

**Objetivo:** Explorar os dados dos alunos de 2022, 2023 e 2024 para entender os padrões de defasagem escolar, a evolução dos indicadores e validar o modelo preditivo treinado.

**Autores:** Bruno Obara — FIAP Machine Learning Fase 5

In [ ]:
import os, json, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

BASE = os.path.abspath('backend')
DATA = os.path.join(BASE, 'data', 'raw')
MODEL_DIR = os.path.join(BASE, 'model')
print('Base dir:', BASE)

## 1. Carregamento e Unificação dos Dados

In [ ]:
def ler_csv(path):
    for enc in ('utf-8', 'latin-1', 'cp1252'):
        try:
            return pd.read_csv(path, sep=';', encoding=enc, dtype=str)
        except UnicodeDecodeError:
            continue

def conv_float(s):
    if s.dtype == object:
        return pd.to_numeric(s.str.replace(',', '.', regex=False), errors='coerce')
    return pd.to_numeric(s, errors='coerce')

# Valores aceitos como "ativo"
VALORES_ATIVOS = {'cursando', 'ativo', 'sim'}

def filtrar_ativos(df):
    """Filtra apenas alunos ativos via coluna 'Ativo/ Inativo'. Se ausente, mantém tudo."""
    for col in ('Ativo/ Inativo', 'Ativo/ Inativo.1'):
        if col in df.columns:
            mask = df[col].astype(str).str.strip().str.lower().isin(VALORES_ATIVOS)
            n_removidos = (~mask).sum()
            df = df[mask].copy()
            print(f"  [filtro ativo] Coluna '{col}': {n_removidos} inativos removidos, {len(df)} ativos mantidos.")
            break
    return df

rename_map = {
    'Idade 22': 'Idade', 'INDE 22': 'INDE',
    'Defas': 'Defasagem', 'Pedra 22': 'Pedra',
    'INDE 2023': 'INDE', 'Pedra 2023': 'Pedra',
    'INDE 2024': 'INDE', 'Pedra 2024': 'Pedra',
}

frames = []
for ano, fname in [(2022, 'dataset_2022.csv'), (2023, 'dataset_2023.csv'), (2024, 'dataset_2024.csv')]:
    df = ler_csv(os.path.join(DATA, fname)).rename(columns=rename_map)
    if ano == 2024:
        print(f"Ano {ano} — aplicando filtro de alunos ativos:")
        df = filtrar_ativos(df)
    df['Ano'] = ano
    frames.append(df)
    print(f"Ano {ano}: {len(df)} registros carregados.")

raw = pd.concat(frames, ignore_index=True)

NUM_COLS = ['INDE', 'IAA', 'IEG', 'IPS', 'IPP', 'IDA', 'IPV', 'IAN', 'Defasagem', 'Idade']
for c in NUM_COLS:
    if c in raw.columns:
        raw[c] = conv_float(raw[c])

# Parse Fase
import re
def parse_fase(v):
    s = str(v).strip().upper()
    if s == 'ALFA': return 0
    m = re.search(r'\d+', s)
    return int(m.group()) if m else np.nan

raw['Fase_num'] = raw['Fase'].apply(parse_fase)
raw['RISCO']    = (raw['Defasagem'] < 0).astype('Int8')

print(f'\nShape total (somente ativos): {raw.shape}')
raw.head(3)


## 2. Visão Geral dos Dados

In [ ]:
print('=== Alunos por Ano ===')
print(raw.groupby('Ano')['RA'].nunique().rename('Qtd Alunos').to_string())

print('\n=== Nulos por coluna (%) ===')
nulos = (raw[NUM_COLS].isnull().sum() / len(raw) * 100).round(1)
print(nulos[nulos > 0].to_string())

raw[[c for c in NUM_COLS if c in raw.columns]].describe().round(2)

## 3. Distribuição do INDE por Ano

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=raw.dropna(subset=['INDE']), x='Ano', y='INDE',
            palette='Blues_d', width=0.5, ax=ax)
ax.set_title('Distribuição do INDE por Ano', fontsize=14, fontweight='bold')
ax.set_xlabel('Ano')
ax.set_ylabel('INDE')
ax.axhline(7.0, ls='--', color='red', lw=1.2, label='Limiar Ametista (7.0)')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Evolução Média dos Indicadores

In [ ]:
indicadores = [c for c in ['INDE', 'IAA', 'IEG', 'IPS', 'IDA', 'IPV', 'IAN'] if c in raw.columns]
evolucao = raw.groupby('Ano')[indicadores].mean().round(3)

fig, ax = plt.subplots(figsize=(10, 5))
cores = ['#6366f1', '#10b981', '#f59e0b', '#ef4444', '#3b82f6', '#8b5cf6', '#ec4899']
for i, ind in enumerate(indicadores):
    ax.plot(evolucao.index, evolucao[ind], marker='o', lw=2.5,
            color=cores[i % len(cores)], label=ind)

ax.set_title('Evolução Média dos Indicadores por Ano', fontsize=14, fontweight='bold')
ax.set_ylabel('Valor Médio (0–10)')
ax.set_xlabel('Ano')
ax.set_ylim(0, 10.5)
ax.legend(loc='lower right', ncol=2)
ax.set_xticks(evolucao.index)
plt.tight_layout()
plt.show()

print(evolucao.to_string())

## 5. Distribuição por Classificação (Pedras)

In [ ]:
pedra_order = ['Quartzo', 'Ágata', 'Ametista', 'Topázio']
pedra_colors = ['#94a3b8', '#38bdf8', '#a855f7', '#fbbf24']

df_pedra = raw[raw['Pedra'].isin(pedra_order)].copy()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Geral
contagem = df_pedra['Pedra'].value_counts().reindex(pedra_order, fill_value=0)
axes[0].bar(contagem.index, contagem.values, color=pedra_colors, edgecolor='white', linewidth=1.2)
axes[0].set_title('Distribuição Geral por Pedra', fontweight='bold')
axes[0].set_ylabel('Nº de Alunos')
for i, v in enumerate(contagem.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold', fontsize=10)

# Por Ano
cross = pd.crosstab(df_pedra['Pedra'], df_pedra['Ano'])
cross = cross.reindex(pedra_order, fill_value=0)
cross.plot(kind='bar', ax=axes[1], colormap='viridis', edgecolor='white', linewidth=0.8)
axes[1].set_title('Distribuição por Pedra e Ano', fontweight='bold')
axes[1].set_ylabel('Nº de Alunos')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Ano')

plt.tight_layout()
plt.show()

## 6. Análise de Risco de Defasagem

In [ ]:
df_risco = raw.dropna(subset=['Defasagem']).copy()
df_risco['Defasagem'] = df_risco['Defasagem'].astype(int)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Distribuição de defasagem
vals = df_risco['Defasagem'].value_counts().sort_index()
colors_def = ['#ef4444' if v < 0 else '#10b981' for v in vals.index]
axes[0].bar(vals.index.astype(str), vals.values, color=colors_def, edgecolor='white')
axes[0].set_title('Distribuição da Defasagem', fontweight='bold')
axes[0].set_xlabel('Defasagem (fases)')
axes[0].set_ylabel('Nº de Alunos')

# % em risco por ano
risco_ano = df_risco.groupby('Ano')['RISCO'].mean() * 100
axes[1].bar(risco_ano.index.astype(str), risco_ano.values,
            color=['#f59e0b', '#ef4444', '#dc2626'], edgecolor='white')
axes[1].set_title('% de Alunos em Risco por Ano', fontweight='bold')
axes[1].set_ylabel('% em Risco')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
for i, (ano, val) in enumerate(risco_ano.items()):
    axes[1].text(i, val + 0.8, f'{val:.1f}%', ha='center', fontweight='bold')

# INDE vs RISCO
sns.boxplot(data=df_risco.dropna(subset=['INDE']), x='RISCO', y='INDE',
            palette={0: '#10b981', 1: '#ef4444'}, ax=axes[2])
axes[2].set_title('INDE × Risco de Defasagem', fontweight='bold')
axes[2].set_xlabel('Risco (0=Não, 1=Sim)')
axes[2].set_ylabel('INDE')

plt.tight_layout()
plt.show()

print('Total com risco:', df_risco['RISCO'].sum(), f'({df_risco["RISCO"].mean()*100:.1f}%)')

## 7. Correlação entre Indicadores

In [ ]:
cols_corr = [c for c in ['INDE', 'IAA', 'IEG', 'IPS', 'IDA', 'IPV', 'IAN', 'RISCO', 'Fase_num'] if c in df_risco.columns]
corr = df_risco[cols_corr].astype(float).corr()

plt.figure(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1,
            linewidths=0.5, square=True, cbar_kws={'shrink': .7})
plt.title('Matriz de Correlação dos Indicadores', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Feature Importance do Modelo

In [ ]:
fi_path = os.path.join(MODEL_DIR, 'feature_importance.json')

if os.path.exists(fi_path):
    with open(fi_path) as f:
        fi_data = json.load(f)
    fi_df = pd.DataFrame(fi_data).sort_values('importance')

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.barh(fi_df['feature'], fi_df['importance'], color='#6366f1', edgecolor='white')
    ax.set_title('Importância das Features — Random Forest', fontsize=13, fontweight='bold')
    ax.set_xlabel('Importância Relativa')
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    for bar, val in zip(bars, fi_df['importance']):
        ax.text(val + 0.003, bar.get_y() + bar.get_height()/2,
                f'{val*100:.1f}%', va='center', fontsize=9)
    plt.tight_layout()
    plt.show()

    print('\nNota: IAN (Índice de Adequação ao Nível) domina a predição pois é')
    print('matematicamente derivado da defasagem — confirma coerência do modelo.')
else:
    print('feature_importance.json não encontrado. Execute backend/src/train.py primeiro.')

## 9. Métricas do Modelo e Matriz de Confusão

In [ ]:
metricas_path = os.path.join(MODEL_DIR, 'metricas.json')

if os.path.exists(metricas_path):
    with open(metricas_path) as f:
        metricas = json.load(f)

    print('=== Métricas do Modelo ===')
    for k in ['accuracy', 'roc_auc', 'f1', 'precision', 'recall']:
        print(f'  {k:12s}: {metricas[k]:.4f}')
    print(f'  Amostras    : treino={metricas["n_treino"]}  teste={metricas["n_teste"]}')

    # Matriz de confusão
    cm = np.array(metricas['confusion_matrix'])
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=['Sem Risco', 'Em Risco']
    )
    disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
    axes[0].set_title('Matriz de Confusão', fontweight='bold')

    # Curva ROC
    fpr = metricas['roc_curve']['fpr']
    tpr = metricas['roc_curve']['tpr']
    axes[1].plot(fpr, tpr, color='#6366f1', lw=2,
                 label=f'ROC-AUC = {metricas["roc_auc"]:.4f}')
    axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
    axes[1].set_xlabel('Taxa de Falsos Positivos')
    axes[1].set_ylabel('Taxa de Verdadeiros Positivos')
    axes[1].set_title('Curva ROC', fontweight='bold')
    axes[1].legend(loc='lower right')
    axes[1].set_xlim([0, 1])
    axes[1].set_ylim([0, 1.02])

    plt.tight_layout()
    plt.show()
else:
    print('metricas.json não encontrado. Execute backend/src/train.py primeiro.')

## 10. Conclusões

### Dados
- Os três anos possuem estrutura similar porém com variações de nome de coluna — o pipeline de pré-processamento normaliza tudo automaticamente.
- A partir de 2024 o dataset inclui a coluna **`Ativo/ Inativo`**. O filtro mantém apenas alunos com status `'Cursando'`, garantindo que inativos não distorçam métricas e predições. No dataset atual todos os 1.156 registros de 2024 estão ativos.
- A coluna **IAN** (Índice de Adequação ao Nível) é matematicamente derivada da relação entre a Fase atual e a Fase ideal, o que explica sua dominância como feature (>83%). O modelo confirma essa coerência.
- Aproximadamente 55% dos alunos apresentam algum grau de defasagem, justificando a necessidade do programa de aceleração.

### Modelo e Deploy
- O **Random Forest** atingiu **ROC-AUC = 1.0** e **Accuracy = 100%** no conjunto de teste. Esse resultado é esperado pois o IAN codifica diretamente a defasagem.
- A solução foi containerizada com **Docker** e deployada no **Render**. 
- **Desafio Técnico:** Durante o deploy, identificamos que o contexto de build do Render exclui arquivos grandes (>100KB) por padrão. A solução foi configurar o `Dockerfile` para baixar os CSVs diretamente do GitHub via `curl` durante a construção da imagem, garantindo a integridade dos dados na API.
- O dashboard React consome dados reais via endpoints de analytics, provendo uma visão transparente do impacto da Associação Passos Mágicos.